# Assignment 2: reasoning and self-reflection
Run the cells in order. Qwen uses GPU T4 x2 with Internet enabled.
One server evaluates naive, planning and reflection agents on BabyAI goto, Crafter and MiniHack MazeWalk 9x9.
Five recorded seeds (42–46), one episode per seed/environment, 50 steps maximum: 45 episodes per model.
This is a bounded comparison, not the complete BALROG benchmark.

Results, tables and three DRAFT submission ZIPs are saved in /kaggle/working.
A cloud model and written trajectory-based reflections are still required before submission.
For your partner: change CLIENT and MODEL in the first cell, add the provider API key as a Kaggle secret, and turn off the GPU.

In [ ]:
from pathlib import Path
import os
import subprocess
import tempfile

CLIENT = "vllm"  # For your partner: "gemini", "openai", or "claude".
MODEL = "Qwen/Qwen2.5-7B-Instruct"  # Set your cloud provider's exact model ID in cloud mode.
PORT = 18080

if CLIENT == "vllm":
    gpus = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], text=True
    ).strip().splitlines()
    if len(gpus) != 2:
        raise RuntimeError("Select GPU T4 x2.")
    print("GPUs:", *gpus, sep="\n- ")
else:
    from kaggle_secrets import UserSecretsClient
    secret_name = {"gemini": "GEMINI_API_KEY", "openai": "OPENAI_API_KEY", "claude": "ANTHROPIC_API_KEY"}[CLIENT]
    os.environ[secret_name] = UserSecretsClient().get_secret(secret_name)
    if MODEL == "Qwen/Qwen2.5-7B-Instruct":
        raise ValueError("Set MODEL to the exact cloud model ID.")

TEMP_ROOT = Path("/kaggle/temp")
TEMP_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(TEMP_ROOT / "huggingface")
PROJECT = Path(tempfile.mkdtemp(prefix="balrog-", dir=TEMP_ROOT)) / "agentic-llm-assignment-2"
subprocess.run([
    "git", "clone", "--branch", "siddique/assignment-foundation", "--single-branch",
    "https://github.com/siddiqueMangi2003/agentic-llm-assignment-2.git", str(PROJECT),
], check=True)
RESULTS = Path("/kaggle/working/assignment_results")
SUBMISSION = Path("/kaggle/working/submission")
print("Repository:", PROJECT)

In [ ]:
import sys

ENV = TEMP_ROOT / "balrog-py310"
subprocess.run([sys.executable, "-m", "pip", "install", "uv"], check=True)
PYTHON = ENV / "bin" / "python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "uv", "venv", "--python", "3.10", str(ENV)], check=True)
packages = ["-e", str(PROJECT), "setuptools<81"]
if CLIENT == "vllm":
    packages.append("vllm==0.31.0")
subprocess.run(
    [sys.executable, "-m", "uv", "pip", "install", "--python", str(PYTHON), *packages],
    check=True,
)
os.environ["PATH"] = str(ENV / "bin") + os.pathsep + os.environ["PATH"]

In [ ]:
import time
from urllib.error import URLError
from urllib.request import urlopen

command = [
    str(PYTHON), "scripts/run_assignment.py",
    "--model", MODEL, "--client", CLIENT,
    "--base-url", f"http://127.0.0.1:{PORT}/v1",
    "--output", str(RESULTS),
]

server = None
log_path = Path("/kaggle/working/vllm.log")
try:
    if CLIENT == "vllm":
        server_cmd = [
            str(ENV / "bin" / "vllm"), "serve", MODEL,
            "--host", "127.0.0.1", "--port", str(PORT),
            "--tensor-parallel-size", "2",
            "--max-model-len", "16384", "--gpu-memory-utilization", "0.85",
        ]
        with log_path.open("w") as log:
            server = subprocess.Popen(server_cmd, stdout=log, stderr=subprocess.STDOUT)
        for _ in range(240):
            if server.poll() is not None:
                raise RuntimeError("vLLM stopped:\n" + log_path.read_text()[-4000:])
            try:
                with urlopen(f"http://127.0.0.1:{PORT}/v1/models", timeout=2):
                    break
            except (URLError, TimeoutError):
                time.sleep(5)
        else:
            raise TimeoutError("vLLM did not start:\n" + log_path.read_text()[-4000:])
    subprocess.run(command, cwd=PROJECT, check=True)
finally:
    if server is not None:
        server.terminate()
        try:
            server.wait(timeout=15)
        except subprocess.TimeoutExpired:
            server.kill()
            server.wait()

In [ ]:
import shutil

# Fetch the packaging fix without rerunning any evaluation.
subprocess.run(["git", "pull", "--ff-only"], cwd=PROJECT, check=True)
archive = shutil.make_archive(
    "/kaggle/working/assignment-results", "zip",
    root_dir=RESULTS.parent, base_dir=RESULTS.name,
)
print("Download raw results:", archive)
subprocess.run([
    str(PYTHON), "scripts/prepare_submission.py",
    "--results", str(RESULTS), "--output", str(SUBMISSION),
], cwd=PROJECT, check=True)
print("Download comparison tables and DRAFT Ass2-1/2/3.zip:", SUBMISSION)